# DIMMS and RINGSS Nightly Summary

Author(s): Bruno Quint
Last Update: 2025-10-24


In [ ]:
day_obs = 20251020
maximum_elevation_angle = 85 # degrees

In [ ]:
# Clustering parameters (optimized for noisy data)
CLUSTERING_EPS = 0.1  # degrees - spatial tolerance for grouping observations
CLUSTERING_MIN_SAMPLES = 20  # minimum observations to form a cluster (increased from 5)

# Quality filtering thresholds
MIN_OBSERVATIONS = 50  # minimum observations to be considered a significant target
MIN_DURATION_MINUTES = 10  # OR minimum duration in minutes

# Shared target matching tolerances
SHARED_TARGET_RA_TOLERANCE = 0.15  # degrees
SHARED_TARGET_DECL_TOLERANCE = 0.15  # degrees

# Temporal overlap detection
SIMULTANEOUS_TIME_WINDOW_MINUTES = 5  # time window for "simultaneous" observations

# Vera Rubin Observatory latitude 
RUBIN_LATITUDE_DEGREES = -30.2446  # degrees

## Setup notebook

In [ ]:
import os    
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astropy.time import Time
from bokeh.io import output_notebook
from bokeh.plotting import figure, output_file, save, show
from bokeh.layouts import gridplot
from bokeh.models import CrosshairTool
from sklearn.cluster import DBSCAN
from datetime import timedelta

from lsst.ts.xml.enums.DIMM import ScopeMotion
from lsst.summit.utils.efdUtils import (
    getDayObsStartTime, 
    getDayObsEndTime, 
    getEfdData, 
    makeEfdClient
)

In [ ]:
efd_client = makeEfdClient()

start_time = getDayObsStartTime(day_obs)
end_time = getDayObsEndTime(day_obs)

topics_and_columns = {
    "lsst.sal.DIMM.status": [
        "ra", "decl", "azimuth", "altitude", "motionState", "salIndex"
    ]
}

### Query data

In [ ]:
# Step 1: Read data and do initial processing
# ============================================================================
dimm_key = "lsst.sal.DIMM.status"
dimm_df = getEfdData(
    client=efd_client, 
    topic=dimm_key,
    columns=topics_and_columns[dimm_key],
    begin=start_time,
    end=end_time
)

# Our data has lots of NaN's. Let's drop them for now.
dimm_df = dimm_df.dropna(how="all")

# Clear out rows where ra/dec are both 0.
dimm_df = dimm_df[dimm_df["ra"] != 0]
dimm_df = dimm_df[dimm_df["decl"] != 0]

# Map the motion state so it is human readable
dimm_df["motionStateName"] = dimm_df["motionState"].map(lambda x: ScopeMotion(x).name)

# Map salIndex to hardware names
hardware_names = {1: 'Tower DIMM', 2: 'Portable DIMM'}
dimm_df['hardware'] = dimm_df['salIndex'].map(hardware_names)

# Filter valid data (non-null RA/DECL)
dimm_df = dimm_df[dimm_df['ra'].notna() & dimm_df['decl'].notna()].copy()

# Target Selection

Let's start ensureing that we are comparing apples to apples.  
By the time we wrote this notebook, the `motionState` had unreliable data.  
The analysis will probably be easier when we have it.  

Let me start looping over a single `day_obs`.  
Then I will make a timeline plot for the two different DIMMs.  
After that, I used [sklearn.cluster.DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) to find out when we are tracking on sky.  
Not that this solution is temporary until we have `motionState` fixed with real values.  
Finally, we show the timeline again with tracking data being highlighted.  

### Filter data based on tracking speed

In [ ]:
def get_max_tracking_speeds(latitude, elevation):
    """
    Returns (max_az_rate, max_el_rate, max_total_speed) in deg/s
    """
    import numpy as np
    
    OMEGA = 0.00417807462  # Earth's rotation in deg/s
    
    lat_rad = np.deg2rad(latitude)
    elev_rad = np.deg2rad(elevation)
    
    max_az = OMEGA / np.cos(elev_rad)
    max_el = OMEGA * np.abs(np.sin(lat_rad))
    max_total = np.sqrt(max_az**2 + max_el**2)
    
    return max_az, max_el, max_total

max_az_speed, max_el_speed, max_total_speed = get_max_tracking_speeds(
    RUBIN_LATITUDE_DEGREES, maximum_elevation_angle)

print(f"Maximum tracking speeds at elevation angle {maximum_elevation_angle} degrees:")
print(f"Max azimuth speed: {max_az_speed:.4f} deg/s")
print(f"Max elevation speed: {max_el_speed:.4f} deg/s")
print(f"Max total speed: {max_total_speed:.4f} deg/s")

In [ ]:
def calculate_az_el_speeds(df):
    """
    Calculate azimuth and elevation tracking speeds from DIMM data.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'azimuth', 'altitude' columns and datetime index
        
    Returns:
    --------
    pd.DataFrame
        Input DataFrame with added columns:
        - 'az_speed': azimuth rate (deg/s)
        - 'el_speed': elevation rate (deg/s)
        - 'angular_speed': total angular speed (deg/s)
    """
    import numpy as np
    
    df = df.copy()
    
    # Initialize
    df['az_speed'] = np.nan
    df['el_speed'] = np.nan
    df['angular_speed'] = np.nan
    
    # Calculate separately for each hardware
    for sal_idx in df['salIndex'].unique():
        mask = df['salIndex'] == sal_idx
        data = df[mask].sort_index()
        
        # Get consecutive differences
        az_diff = data['azimuth'].diff().values
        alt_diff = data['altitude'].diff().values
        time_diff = data.index.to_series().diff().dt.total_seconds().values
        
        # Avoid division by zero
        time_diff[time_diff == 0] = np.nan
        
        # Calculate speeds (deg/s)
        az_speed = np.abs(az_diff) / time_diff
        el_speed = np.abs(alt_diff) / time_diff
        total_speed = np.sqrt(az_speed**2 + el_speed**2)
        
        # Assign back
        df.loc[mask, 'az_speed'] = az_speed
        df.loc[mask, 'el_speed'] = el_speed
        df.loc[mask, 'angular_speed'] = total_speed
    
    return df

# Calculate azimuth and elevation speeds
dimm_df = calculate_az_el_speeds(dimm_df)

In [ ]:
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

# Radial azimuth/elevation tower dimm - unfiltered data
tdimm_df = dimm_df[dimm_df['salIndex'] == 1]
pdimm_df = dimm_df[dimm_df['salIndex'] == 2]

t_speed_mask = (tdimm_df["az_speed"] <= max_az_speed) & (tdimm_df["el_speed"] <= max_el_speed)
p_speed_mask = (pdimm_df["az_speed"] <= max_az_speed) & (pdimm_df["el_speed"] <= max_el_speed)

ax0.plot(tdimm_df.index, tdimm_df["azimuth"], 'C0-', alpha=0.2)
ax0.plot(pdimm_df.index, pdimm_df["azimuth"], 'C1-', alpha=0.2)

ax1.plot(tdimm_df.index, tdimm_df["altitude"], 'C0-', alpha=0.2)
ax1.plot(pdimm_df.index, pdimm_df["altitude"], 'C1-', alpha=0.2)

ax0.plot(tdimm_df[t_speed_mask].index, tdimm_df["azimuth"][t_speed_mask], 'C0o', alpha=0.5, label='Tower DIMM')
ax0.plot(pdimm_df[p_speed_mask].index, pdimm_df["azimuth"][p_speed_mask], 'C1o', alpha=0.5, label='Portable DIMM')
ax0.legend()

ax1.plot(tdimm_df[t_speed_mask].index, tdimm_df["altitude"][t_speed_mask], 'C0o', alpha=0.5)
ax1.plot(pdimm_df[p_speed_mask].index, pdimm_df["altitude"][p_speed_mask], 'C1o', alpha=0.5)

ax0.grid(":", alpha=0.5)
ax0.set_ylabel("Azimuth [deg]")
ax1.set_ylabel("Elevation [deg]")
ax1.set_xlabel("Time [UTC]")
ax1.grid(":", alpha=0.5)

fig.suptitle("Tower and Portable DIMM\nData filtered using tracking speed")

plt.show()

### Cluster targets for each DIMM separately

In [ ]:
print("\n" + "="*80)
print("CLUSTERING TARGETS BY HARDWARE")
print("="*80)
print(f"\nParameters: eps={CLUSTERING_EPS}°, min_samples={CLUSTERING_MIN_SAMPLES}")
print(f"Quality filters: ≥{MIN_OBSERVATIONS} obs OR ≥{MIN_DURATION_MINUTES} min duration")


def cluster_targets(data, hardware_name, eps=CLUSTERING_EPS, min_samples=CLUSTERING_MIN_SAMPLES):
    """
    Cluster targets using DBSCAN algorithm with robust parameters
    
    Parameters:
    -----------
    data : DataFrame
        Tracking data for a single hardware
    hardware_name : str
        Name of the hardware (for display)
    eps : float
        Maximum distance between points in same cluster (degrees)
    min_samples : int
        Minimum points to form a cluster
    
    Returns:
    --------
    DataFrame with target_id column added, dict with statistics
    """
    print(f"\n{hardware_name}:")
    print(f"  Records to cluster: {len(data):,}")
    
    # Prepare coordinates
    coords = data[['ra', 'decl']].values
    
    # Perform clustering
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    
    # Add cluster labels
    data_clustered = data.copy()
    data_clustered['target_id'] = clustering.labels_
    
    # Initial statistics
    n_clusters_initial = len(set(clustering.labels_)) - (1 if -1 in clustering.labels_ else 0)
    n_noise_initial = sum(clustering.labels_ == -1)
    
    print(f"  Initial clusters identified: {n_clusters_initial}")
    print(f"  Initial noise points: {n_noise_initial}")
    
    # Calculate cluster statistics for filtering
    valid_data = data_clustered[data_clustered['target_id'] >= 0]
    
    if len(valid_data) > 0:
        cluster_stats = valid_data.groupby('target_id').agg({
            'ra': 'count'
        })
        cluster_stats.columns = ['count']
        
        # Calculate durations
        time_info = valid_data.groupby('target_id').apply(
            lambda x: (x.index.max() - x.index.min()).total_seconds() / 60,
            include_groups=False
        )
        cluster_stats['duration_min'] = time_info
        
        # Apply quality filters
        significant_mask = (
            (cluster_stats['count'] >= MIN_OBSERVATIONS) | 
            (cluster_stats['duration_min'] >= MIN_DURATION_MINUTES)
        )
        significant_cluster_ids = cluster_stats[significant_mask].index.tolist()
        
        # Filter the data to keep only significant clusters
        # Mark filtered clusters as noise (-1)
        data_clustered.loc[
            (data_clustered['target_id'] >= 0) & 
            (~data_clustered['target_id'].isin(significant_cluster_ids)),
            'target_id'
        ] = -1
        
        # Renumber clusters sequentially starting from 0
        old_to_new = {old_id: new_id for new_id, old_id in enumerate(sorted(significant_cluster_ids))}
        data_clustered.loc[data_clustered['target_id'] >= 0, 'target_id'] = \
            data_clustered.loc[data_clustered['target_id'] >= 0, 'target_id'].map(old_to_new)
        
        n_clusters_final = len(significant_cluster_ids)
        n_noise_final = sum(data_clustered['target_id'] == -1)
        n_filtered = n_clusters_initial - n_clusters_final
        
        print(f"  Clusters after filtering: {n_clusters_final}")
        print(f"  Filtered out: {n_filtered} small/short clusters")
        print(f"  Final noise/filtered points: {n_noise_final}")
        
        # Statistics for filtered clusters
        small_clusters = cluster_stats[~significant_mask]
        if len(small_clusters) > 0:
            print(f"  Filtered cluster sizes: {small_clusters['count'].min():.0f} to {small_clusters['count'].max():.0f} obs")
            print(f"  Filtered observations: {small_clusters['count'].sum():.0f}")
    
    stats = {
        'initial_clusters': n_clusters_initial,
        'final_clusters': n_clusters_final if len(valid_data) > 0 else 0,
        'filtered_clusters': n_filtered if len(valid_data) > 0 else 0,
        'noise_points': n_noise_final if len(valid_data) > 0 else n_noise_initial
    }
    
    return data_clustered, stats


# Cluster each hardware separately
tower_data, tower_stats = cluster_targets(
    dimm_df[dimm_df['salIndex'] == 1],
    'Tower DIMM (salIndex=1)'
)

portable_data, portable_stats = cluster_targets(
    dimm_df[dimm_df['salIndex'] == 2],
    'Portable DIMM (salIndex=2)'
)

# Combine back into single dataframe with hardware-specific target IDs
tower_data['global_target_id'] = tower_data['target_id'].apply(
    lambda x: f"T{x}" if x >= 0 else "T-1"
)
portable_data['global_target_id'] = portable_data['target_id'].apply(
    lambda x: f"P{x}" if x >= 0 else "P-1"
)

dimm_df = pd.concat([tower_data, portable_data])
dimm_df = dimm_df.sort_index()

# Record the data into a CSV file (optional)
dimm_df.to_csv(f"dimm_df_{day_obs}.csv")